In [0]:
import os
import json
import shutil
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# 1. PROJECT CONFIGURATION
PROJECT_FOLDER = Path("/Volumes/workspace/default/py_de_ai_project")
LANDING = PROJECT_FOLDER / "landing"
DATA = PROJECT_FOLDER / "data"
PROCESSING = DATA / "processing"
ARCHIVE = DATA / "archive"
ERROR = DATA / "error"
OUTPUT = DATA / "output"

# Create folders if they don't exist
for folder in [LANDING, PROCESSING, ARCHIVE, ERROR, OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

# Load API key and initialize AI client
load_dotenv(PROJECT_FOLDER / ".env")
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"

print("Folders created and AI client initialized!")

In [0]:
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_FOLDER = Path("/Volumes/workspace/default/py_de_ai_project")
ENV_FILE = PROJECT_FOLDER / ".env"

print("Project folder exists:", PROJECT_FOLDER.exists())
print(".env file exists:", ENV_FILE.exists())

load_dotenv(ENV_FILE, override=True)

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", bool(api_key))

In [0]:
import sys
import google
import importlib.util
import importlib.metadata

print("Python version:", sys.version)
print("Google package location:", google.__path__)

print("GenAI module:", importlib.util.find_spec("google.genai"))

print("Installed package version:",
      importlib.metadata.version("google-genai"))

print("Python environment:", sys.executable)

In [0]:
%pip show -f google-genai

In [0]:
import sys
import importlib

PACKAGE_PATH = "/local_disk0/.ephemeral_nfs/envs/pythonEnv-f1bb79b2-e982-47bd-8d77-a1f1b76b1900/lib/python3.12/site-packages"

# Add installed package location to Python search path
sys.path.insert(0, PACKAGE_PATH)

# Refresh Python module discovery
importlib.invalidate_caches()

# Import Gemini SDK
from google import genai

print("Google GenAI imported successfully!")
print(genai.__file__)

In [0]:
%pip install google-genai

In [0]:
from google import genai

print("Google GenAI imported successfully")

In [0]:
# 2. DEFINE FUNCTION
def load_dataframe(file_path):
    extension = file_path.suffix.lower()

    if extension == ".csv":
        return pd.read_csv(file_path)
    if extension == ".json":
        return pd.read_json(file_path)
    if extension == ".xlsx":
        return pd.read_excel(file_path)
    if extension == ".txt":
        return pd.read_csv(file_path, sep="|")

    print(f"Unsupported file type: {extension}")
    return None

# TEST IT
test_file = LANDING / "employee.csv" # Change to your actual file name
df = load_dataframe(test_file)

print("\n--- Testing load_dataframe ---")
print(df.head())

In [0]:
# 3. DEFINE FUNCTION
def profile_dataframe(df, file_name):
    row_count = len(df)
    if row_count:
        null_percentage = (df.isnull().sum() / row_count * 100).round(2)
    else:
        null_percentage = df.isnull().sum()

    column_profile = pd.DataFrame({
        "column_name": df.columns,
        "data_type": df.dtypes.astype(str).values,
        "null_count": df.isnull().sum().values,
        "actual_count": row_count,
        "unique_count": df.nunique().values,
        "null_percentage": null_percentage.values
    })
 
    profile_file = OUTPUT / f"{Path(file_name).stem}_profile.csv"
    column_profile.to_csv(profile_file, index=False)
    data_types = df.dtypes.astype(str).to_dict()
    duplicates = df.duplicated().sum()
 
    return data_types, column_profile, duplicates

# TEST IT
print("\n--- Testing profile_dataframe ---")
data_types, column_profile, duplicates = profile_dataframe(df, test_file.name)

print("Data Types Dictionary:")
print(data_types)
print("\nColumn Profile DataFrame:")
print(column_profile)

In [0]:
# 4. DEFINE FUNCTION
def build_prompts(file_name, df, data_types, column_profile):
    sample_json = df.head(10).astype(str).to_json(orient="records", indent=2, force_ascii=False)
    columns = df.columns.tolist()
    types_json = json.dumps(data_types, indent=2, ensure_ascii=False)
    profile_text = column_profile.to_string(index=False)

    classification_prompt = f"""File: {file_name}
Columns: {json.dumps(columns)}
Types:
{types_json}
Profile:
{profile_text}
Sample:
{sample_json}
Determine: Data governance domain, PII/confidential tags, automated data quality rules & checks. Return valid JSON."""

    schema_prompt = f"""File: {file_name}
Columns: {json.dumps(columns)}
Types:
{types_json}
Profile:
{profile_text}
Sample:
{sample_json}
Determine: Target analytical DDL schema, primary/foreign key evidence & data quality constraints. Return valid JSON."""
    return classification_prompt, schema_prompt

# TEST IT
print("\n--- Testing build_prompts ---")
c_prompt, s_prompt = build_prompts(test_file.name, df, data_types, column_profile)

print("Classification Prompt preview:")
print(c_prompt[:500] + "...\n[TRUNCATED FOR DISPLAY]")

In [0]:
from pathlib import Path

PROJECT_FOLDER = Path("/Volumes/workspace/default/py_de_ai_project")

LANDING = PROJECT_FOLDER / "landing"
DATA = PROJECT_FOLDER / "data"
PROCESSING = DATA / "processing"
ARCHIVE = DATA / "archive"
ERROR = DATA / "error"
OUTPUT = DATA / "output"
SCRIPT = PROJECT_FOLDER / "script"

print("Project folder:", PROJECT_FOLDER.exists())
print("Landing folder:", LANDING.exists())
print("Data folder:", DATA.exists())
print("Script folder:", SCRIPT.exists())

In [0]:
print("LANDING FILES:")
display(dbutils.fs.ls(str(LANDING)))

print("SCRIPT FILES:")
display(dbutils.fs.ls(str(SCRIPT)))

print("DATA FILES:")
display(dbutils.fs.ls(str(DATA)))

In [0]:
import os
from dotenv import load_dotenv

ENV_FILE = PROJECT_FOLDER / ".env"

print("ENV file exists:", ENV_FILE.exists())

load_dotenv(ENV_FILE, override=True)

api_key = os.getenv("GEMINI_API_KEY")

print("Gemini API key loaded:", bool(api_key))

In [0]:
import sys
import runpy

SCRIPT_PATH = str(SCRIPT / "data_profiler.py")

sys.argv = [SCRIPT_PATH, str(PROJECT_FOLDER)]

runpy.run_path(SCRIPT_PATH, run_name="__main__")

In [0]:
print("OUTPUT FILES:")
display(dbutils.fs.ls(str(OUTPUT)))

print("ARCHIVE FILES:")
display(dbutils.fs.ls(str(ARCHIVE)))

print("ERROR FILES:")
display(dbutils.fs.ls(str(ERROR)))